<div align="center">

<img src="https://img.shields.io/badge/ECS-Veri%20Bilimi%20%26%20YZ%20Uzmanl%C4%B1%C4%9F%C4%B1-5B2C1E?style=for-the-badge&logo=python&logoColor=white" alt="ECS VB&YZ 90"/>

# Hafta 9: MLOps Pipeline

**MAKİNE ÖĞRENMESİ UZMANLIĞI** · Modül 9 · 6 Saat

---

<a href="https://colab.research.google.com/github/DrMuratAltun/VB-YZ-90/blob/main/notebooks/hafta09/hafta09_mlops_pipeline.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Colab'da Aç"/></a>&nbsp;
<a href="https://github.com/DrMuratAltun/VB-YZ-90/blob/main/notebooks/hafta09/hafta09_mlops_pipeline.ipynb"><img src="https://img.shields.io/badge/GitHub'da%20A%C3%A7-181717?style=flat&logo=github&logoColor=white" alt="GitHub'da Aç"/></a>&nbsp;
<a href="https://raw.githubusercontent.com/DrMuratAltun/VB-YZ-90/main/web/public/sunumlar/hafta09_mlops_capstone.pdf"><img src="https://img.shields.io/badge/PDF%20Sunum-EC1C24?style=flat&logo=adobeacrobatreader&logoColor=white" alt="PDF Sunum"/></a>&nbsp;
<a href="https://drmurataltun.github.io/VB-YZ-90/hafta/09/"><img src="https://img.shields.io/badge/Web%20Sitesi-2B7A78?style=flat&logo=googlechrome&logoColor=white" alt="Web Sitesi"/></a>

</div>

---

**Eğitmen:** Dr. Murat Altun · [yapayzekaokulum.com](https://yapayzekaokulum.com) · [GitHub](https://github.com/DrMuratAltun)

**Program:** ECS Veri Bilimi ve Yapay Zeka Uzmanlığı · 90 Saat · 15 Hafta
---

> **Bu defterde neler öğreneceksiniz?**
>
> - Model yaşam döngüsü yönetimi
> - joblib ile model kaydetme/yükleme
> - MLOps pipeline tasarımı

# Hafta 9 - MLOps Pipeline

Bu derste:
- Bir makine öğrenmesi modeli eğitip kaydedeceğiz
- `joblib` ile model serileştirme yapacağız
- `sklearn.pipeline.Pipeline` ile uçtan uca iş akışı kuracağız
- Model versiyonlama kavramını öğreneceğiz
- Model kayıt defteri (Model Registry) kavramını tanıyacağız

## 1. Kütüphanelerin Yüklenmesi

### Kütüphanelerin Yüklenmesi

Projede kullanacağımız kütüphaneleri içe aktarıyoruz:

| Kütüphane | Amacı |
|-----------|-------|
| `datetime` | Yardımcı kütüphane |
| `joblib` | Model kaydetme/yükleme |
| `json` | JSON veri formatı işleme |
| `numpy` | Sayısal hesaplamalar ve dizi işlemleri |
| `os` | İşletim sistemi işlemleri |
| `pandas` | Veri çerçeveleri (DataFrame) ile veri analizi |
| `sklearn` | Makine öğrenmesi algoritmaları ve araçları |
| `warnings` | Uyarı mesajlarını yönetme |


In [ ]:
import numpy as np
import pandas as pd
import joblib
import os
import json
from datetime import datetime

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

import warnings
warnings.filterwarnings('ignore')

## 2. Veri Hazırlama ve Model Eğitimi

### Veri Setinin Yüklenmesi

Aşağıdaki kodda veri setini yüklüyoruz ve temel bilgilerine (boyut, sütunlar, ilk satırlar) bakıyoruz. Bu adım her veri bilimi projesinin başlangıcıdır.

In [ ]:
# Iris veri setini yükle
iris = load_iris()
X = iris.data
y = iris.target
feature_names = iris.feature_names
target_names = iris.target_names

print(f"Özellikler: {feature_names}")
print(f"Sınıflar: {target_names}")
print(f"Veri boyutu: {X.shape}")

### Eğitim ve Test Setlerine Ayırma

Veriyi eğitim ve test olarak ikiye bölüyoruz. `stratify` parametresi, her iki sette de sınıf dağılımının aynı kalmasını sağlar. `random_state` ile tekrarlanabilir sonuçlar elde ediyoruz.

In [ ]:
# Eğitim/test ayırma
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Eğitim seti: {X_train.shape[0]} örnek")
print(f"Test seti: {X_test.shape[0]} örnek")

### Model Eğitimi

Aşağıdaki kodda modeli eğitim verisi üzerinde eğitiyoruz (`.fit()`). Eğitim sonrası test verisi üzerinde tahmin yapıp (`.predict()`) başarı metriklerini hesaplıyoruz.

In [ ]:
# Model eğitimi
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Değerlendirme
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Doğruluk: {accuracy:.4f}")
print(f"\nSınıflandırma Raporu:")
print(classification_report(y_test, y_pred, target_names=target_names))

## 3. Model Kaydetme (joblib)

Eğitilmiş modeli diske kaydetmek, modeli her seferinde yeniden eğitmekten kurtarır.

In [ ]:
# Model kaydetme dizini
os.makedirs('models', exist_ok=True)

# Modeli kaydet
model_path = 'models/model.joblib'
joblib.dump(model, model_path)

file_size = os.path.getsize(model_path) / 1024
print(f"Model kaydedildi: {model_path}")
print(f"Dosya boyutu: {file_size:.1f} KB")

## 4. Model Yükleme ve Tahmin

### Model Değerlendirme

Modelin performansını çeşitli metriklerle değerlendiriyoruz: doğruluk (accuracy), kesinlik (precision), duyarlılık (recall) ve F1-skoru.

In [ ]:
# Modeli yükle
loaded_model = joblib.load(model_path)

# Yüklenen model ile tahmin
y_pred_loaded = loaded_model.predict(X_test)
accuracy_loaded = accuracy_score(y_test, y_pred_loaded)
print(f"Yüklenen model doğruluğu: {accuracy_loaded:.4f}")
print(f"Orijinal model ile aynı mı: {np.array_equal(y_pred, y_pred_loaded)}")

## 5. Tahmin Fonksiyonu Oluşturma

### Özellik Seçimi ve Mühendisliği

Model için kullanılacak özellikleri belirliyoruz. Doğru özellik seçimi model performansını doğrudan etkiler.

In [ ]:
def predict_iris(sepal_length, sepal_width, petal_length, petal_width, model_path='models/model.joblib'):
    """Iris çiçeği türünü tahmin eder.
    
    Args:
        sepal_length: Çanak yaprak uzunluğu (cm)
        sepal_width: Çanak yaprak genişliği (cm)
        petal_length: Taç yaprak uzunluğu (cm)
        petal_width: Taç yaprak genişliği (cm)
        model_path: Kaydedilmiş model dosyası yolu
    
    Returns:
        dict: Tahmin sonucu ve olasılıklar
    """
    model = joblib.load(model_path)
    
    features = np.array([[sepal_length, sepal_width, petal_length, petal_width]])
    prediction = model.predict(features)[0]
    probabilities = model.predict_proba(features)[0]
    
    target_names = ['setosa', 'versicolor', 'virginica']
    
    result = {
        'tahmin': target_names[prediction],
        'olasılıklar': {
            name: f"{prob:.2%}" for name, prob in zip(target_names, probabilities)
        }
    }
    return result

# Test
result = predict_iris(5.1, 3.5, 1.4, 0.2)
print("Tahmin Sonucu:")
print(json.dumps(result, indent=2, ensure_ascii=False))

### Farklı bir örnek

Aşağıdaki kod bloğunda bu işlemi gerçekleştiriyoruz.

In [ ]:
# Farklı bir örnek
result2 = predict_iris(6.7, 3.1, 5.6, 2.4)
print("Tahmin Sonucu:")
print(json.dumps(result2, indent=2, ensure_ascii=False))

## 6. sklearn Pipeline: Ön İşleme + Model

Pipeline, veri ön işleme ve model eğitimini tek bir nesnede birleştirir. Bu sayede:
- Veri sızıntısı (data leakage) önlenir
- Tüm iş akışı tek dosyada kaydedilebilir
- Üretim ortamında tutarlılık sağlanır

In [ ]:
# Pipeline oluştur: StandardScaler + RandomForest
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

# Pipeline'ı eğit
pipeline.fit(X_train, y_train)

# Değerlendirme
y_pred_pipe = pipeline.predict(X_test)
accuracy_pipe = accuracy_score(y_test, y_pred_pipe)
print(f"Pipeline doğruluğu: {accuracy_pipe:.4f}")

# Pipeline adımları
print(f"\nPipeline adımları:")
for name, step in pipeline.steps:
    print(f"  {name}: {type(step).__name__}")

### Model Kaydetme

Eğitilmiş modeli diske kaydediyoruz. Böylece modeli tekrar eğitmeden doğrudan yükleyip tahmin yapabiliriz.

In [ ]:
# Tüm pipeline'ı kaydet
pipeline_path = 'models/iris_pipeline.joblib'
joblib.dump(pipeline, pipeline_path)

file_size = os.path.getsize(pipeline_path) / 1024
print(f"Pipeline kaydedildi: {pipeline_path}")
print(f"Dosya boyutu: {file_size:.1f} KB")

### Model Değerlendirme

Modelin performansını çeşitli metriklerle değerlendiriyoruz: doğruluk (accuracy), kesinlik (precision), duyarlılık (recall) ve F1-skoru.

In [ ]:
# Pipeline'ı yükle ve test et
loaded_pipeline = joblib.load(pipeline_path)
y_pred_loaded_pipe = loaded_pipeline.predict(X_test)
print(f"Yüklenen pipeline doğruluğu: {accuracy_score(y_test, y_pred_loaded_pipe):.4f}")

## 7. Model Versiyonlama

### Neden Versiyonlama?

Üretim ortamında birden fazla model sürümü olabilir. Versiyonlama şu soruları yanıtlar:
- Hangi model şu an üretimde?
- Önceki sürüme nasıl geri dönülür?
- Hangi veri ile eğitildi?
- Performansı nasıl değişti?

### Basit Versiyonlama Stratejisi

In [ ]:
def save_model_versioned(model, model_name, version, metrics, notes=""):
    """Modeli versiyon bilgisiyle birlikte kaydeder."""
    
    # Dizin oluştur
    model_dir = f'models/{model_name}/v{version}'
    os.makedirs(model_dir, exist_ok=True)
    
    # Modeli kaydet
    model_file = f'{model_dir}/model.joblib'
    joblib.dump(model, model_file)
    
    # Meta veri kaydet
    metadata = {
        'model_name': model_name,
        'version': version,
        'created_at': datetime.now().isoformat(),
        'metrics': metrics,
        'notes': notes,
        'model_type': type(model).__name__ if not hasattr(model, 'steps') else 'Pipeline',
        'file_path': model_file
    }
    
    meta_file = f'{model_dir}/metadata.json'
    with open(meta_file, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)
    
    print(f"Model kaydedildi: {model_file}")
    print(f"Meta veri: {meta_file}")
    return metadata

# Versiyon 1: Temel model
metrics_v1 = {'accuracy': float(accuracy), 'test_size': len(X_test)}
save_model_versioned(model, 'iris_classifier', '1.0', metrics_v1, 
                     notes='İlk versiyon - RandomForest 100 ağaç')

### Model Kaydetme

Eğitilmiş modeli diske kaydediyoruz. Böylece modeli tekrar eğitmeden doğrudan yükleyip tahmin yapabiliriz.

In [ ]:
# Versiyon 2: Pipeline model
metrics_v2 = {'accuracy': float(accuracy_pipe), 'test_size': len(X_test)}
save_model_versioned(pipeline, 'iris_classifier', '2.0', metrics_v2, 
                     notes='StandardScaler eklenmiş Pipeline versiyon')

## 8. Model Kayıt Defteri (Model Registry) Kavramı

### Model Registry Nedir?

Model Registry, modellerin yaşam döngüsünü yöneten merkezi bir depodur. Gerçek dünyada **MLflow**, **DVC**, **Weights & Biases** gibi araçlar kullanılır.

### Temel Özellikler

| Özellik | Açıklama |
|---------|----------|
| **Versiyonlama** | Her model sürümü benzersiz şekilde tanımlanır |
| **Aşama Yönetimi** | Staging → Production → Archived geçişleri |
| **Meta Veri** | Eğitim parametreleri, metrikler, veri bilgisi |
| **Karşılaştırma** | Farklı sürümlerin performans karşılaştırması |
| **Geri Alma** | Sorunlu bir sürümden öncekine hızlı dönüş |

### MLOps İş Akışı

```
Veri Toplama → Veri İşleme → Model Eğitimi → Model Değerlendirme
     ↓              ↓              ↓               ↓
  DVC/S3      Pipeline/Feature   Experiment     Metrics
              Store              Tracking       Comparison
                                    ↓
                            Model Registry
                                    ↓
                         Staging → Production
                                    ↓
                         Monitoring & Alerting
```

### Popüler MLOps Araçları

| Araç | Kullanım Alanı | Açık Kaynak |
|------|----------------|-------------|
| **MLflow** | Experiment tracking, model registry | Evet |
| **DVC** | Veri versiyonlama, pipeline | Evet |
| **Weights & Biases** | Experiment tracking, görselleştirme | Hayır (freemium) |
| **Kubeflow** | Kubernetes üzerinde ML pipeline | Evet |
| **BentoML** | Model serving/deployment | Evet |

In [ ]:
# Basit bir model registry simülasyonu
class SimpleModelRegistry:
    """Basit model kayıt defteri."""
    
    def __init__(self, registry_dir='models/registry'):
        self.registry_dir = registry_dir
        os.makedirs(registry_dir, exist_ok=True)
        self.registry_file = f'{registry_dir}/registry.json'
        self.registry = self._load_registry()
    
    def _load_registry(self):
        if os.path.exists(self.registry_file):
            with open(self.registry_file, 'r') as f:
                return json.load(f)
        return {'models': {}}
    
    def _save_registry(self):
        with open(self.registry_file, 'w', encoding='utf-8') as f:
            json.dump(self.registry, f, indent=2, ensure_ascii=False)
    
    def register(self, name, version, model, metrics, stage='staging'):
        """Modeli kayıt defterine ekler."""
        model_path = f'{self.registry_dir}/{name}_v{version}.joblib'
        joblib.dump(model, model_path)
        
        if name not in self.registry['models']:
            self.registry['models'][name] = []
        
        entry = {
            'version': version,
            'stage': stage,
            'metrics': metrics,
            'model_path': model_path,
            'registered_at': datetime.now().isoformat()
        }
        self.registry['models'][name].append(entry)
        self._save_registry()
        print(f"✓ {name} v{version} kaydedildi (Aşama: {stage})")
    
    def promote(self, name, version, new_stage):
        """Modelin aşamasını değiştirir."""
        if name in self.registry['models']:
            for entry in self.registry['models'][name]:
                if entry['version'] == version:
                    old_stage = entry['stage']
                    entry['stage'] = new_stage
                    self._save_registry()
                    print(f"✓ {name} v{version}: {old_stage} → {new_stage}")
                    return
        print(f"Model bulunamadı: {name} v{version}")
    
    def list_models(self):
        """Tüm modelleri listeler."""
        print("\n" + "=" * 60)
        print("MODEL KAYIT DEFTERİ")
        print("=" * 60)
        for name, versions in self.registry['models'].items():
            print(f"\n{name}:")
            for v in versions:
                print(f"  v{v['version']} [{v['stage']}] - Doğruluk: {v['metrics'].get('accuracy', 'N/A')}")
        print()

# Registry kullanımı
registry = SimpleModelRegistry()

# Model kayıtları
registry.register('iris_classifier', '1.0', model, 
                   {'accuracy': float(accuracy)}, stage='staging')
registry.register('iris_classifier', '2.0', pipeline, 
                   {'accuracy': float(accuracy_pipe)}, stage='staging')

# v2.0'ı production'a taşı
registry.promote('iris_classifier', '2.0', 'production')

# Tüm modelleri listele
registry.list_models()

## Özet

Bu derste öğrendiklerimiz:
- **joblib** ile model kaydetme ve yükleme
- **sklearn Pipeline** ile ön işleme + model birleştirme
- **Model versiyonlama** ile sürüm takibi
- **Model Registry** kavramı ve yaşam döngüsü yönetimi
- MLOps araçları ve iş akışları

### Alıştırma
1. Farklı bir model (SVM, GradientBoosting) ile pipeline oluşturun ve kaydedin
2. Pipeline'a `PCA` adımı ekleyin (boyut indirgeme)
3. MLflow kurulumu yapıp experiment tracking deneyin: `!pip install mlflow`

---

<div align="center">

<img src="https://img.shields.io/badge/ECS-Veri%20Bilimi%20%26%20YZ-5B2C1E?style=flat-square&logo=python&logoColor=white" alt="ECS"/>

**Dr. Murat Altun** · Veri Bilimi ve Yapay Zeka Eğitmeni

<a href="https://yapayzekaokulum.com">Yapay Zeka Okulum</a> ·
<a href="https://gencyz.com">GençYZ</a> ·
<a href="https://yz-araclari.com">YZ Araçları</a> ·
<a href="https://scholargent.com">ScholarAI</a> ·
<a href="https://drmurataltun.github.io">Kişisel Site</a>

<a href="https://drmurataltun.github.io/VB-YZ-90/">drmurataltun.github.io/VB-YZ-90</a>

---

*Bu materyal ECS Veri Bilimi ve Yapay Zeka Uzmanlığı Programı için hazırlanmıştır.*

&copy; 2026 Dr. Murat Altun. Tüm hakları saklıdır.

</div>